Importing the dependencies

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import scale , StandardScaler
from sklearn.model_selection import train_test_split , GridSearchCV , cross_val_score
from sklearn.metrics import accuracy_score , mean_squared_error , r2_score , confusion_matrix , classification_report , RocCurveDisplay, roc_auc_score , roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold
import warnings
warnings.simplefilter(action='ignore')
sns.set()
plt.style.use("ggplot")
%matplotlib inline

loading the dataset

In [ ]:
df = pd.read_csv("/content/diabetis.csv")

EDA

In [ ]:
df.head(5)

In [ ]:
# supervised or unsupervised
# ans = supervised why? it has a target column_or_1
# regression or classification
# ans : classification


In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
# checking the distribution of the ut come variables
df["Outcome"].value_counts()
#  or (df["Outcome"].value_counts()*100/len(df))

In [ ]:
# ploting the histogram of the age variable
plt.figure(figsize=(7,7))
plt.xlabel('Age', fontsize=10)
plt.ylabel('Count', fontsize=10)
df['Age'].hist(edgecolor = 'black')

In [ ]:
# checking the max and min of the age variable
df['Age'].max()

In [ ]:
df['Age'].min()

In [ ]:
print("MAX AGE IS :"' ' +str(df['Age'].max()))
print("MIN AGE IS :"' ' +str(df['Age'].min()))

In [ ]:
df.columns

In [ ]:
# density graph
fig,ax = plt.subplots(4,2, figsize=(20,20))
sns.distplot(df.Pregnancies, bins=20, ax=ax[0,0], color='red')
sns.distplot(df.Glucose, bins=20, ax=ax[0,1], color='red')
sns.distplot(df.BloodPressure, bins=20, ax=ax[1,0], color='red')
sns.distplot(df.SkinThickness, bins=20, ax=ax[1,1], color='red')
sns.distplot(df.Insulin, bins=20, ax=ax[2,0], color='red')
sns.distplot(df.BMI, bins=20, ax=ax[2,1], color='red')
sns.distplot(df.DiabetesPedigreeFunction, bins=20, ax=ax[3,0], color='red')
sns.distplot(df.Age, bins=20, ax=ax[3,1], color='red')

In [ ]:
df.columns

In [ ]:
# how to check for the min,max,mean...... of each column grouped on the outcome
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
        'BMI', 'DiabetesPedigreeFunction', 'Age']

df.groupby("Outcome")[cols].agg('max')


In [ ]:
# 0 = Healthy
# 1 = Diabetic

f,ax=plt.subplots(1,2,figsize=(10,6))
df['Outcome'].value_counts().plot.pie(explode=[0,0.1],autopct='%1.1f%%',ax=ax[0],shadow=True)
ax[0].set_title('target')
ax[0].set_ylabel('')
sns.countplot(x='Outcome',data=df,ax=ax[1])
ax[1].set_title('target')
plt.show()

In [ ]:
f,ax=plt.subplots(figsize=[20,15])
sns.heatmap(df.corr(),annot=True, fmt = '.2f',ax=ax, cmap='magma')
ax.set_title('Correlation Matrix', fontsize=20)
plt.show()

Data preprocessing

In [ ]:
#checking for missing values
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
  # changing 0 values to nan
  df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']] = df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']].replace(0,np.nan)

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
import missingno as msno
msno.bar(df,color='black')

In [ ]:
# replacing the missing values with the median
def median_target(col):
    return df.groupby('Outcome')[col].median()

columns = [col for col in df.columns if col != 'Outcome']

for col in columns:
    medians = median_target(col)
    df.loc[(df['Outcome'] == 0) & (df[col].isnull()), col] = medians[0]
    df.loc[(df['Outcome'] == 1) & (df[col].isnull()), col] = medians[1]


In [ ]:
df.head()

In [ ]:
 df.isnull().sum()

In [ ]:
# pairplot
p = sns.pairplot(df, hue="Outcome")

In [ ]:
# outlier detection using IQF
for feauture in df:
    Q1 = df[feauture].quantile(0.25)
    Q3 = df[feauture].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1-1.5*IQR
    upper = Q3+1.5*IQR
    if df [(df[feauture]>upper)].any(axis=None):
      print(feauture,"yes")
    else:
      print(feauture,"no")

In [ ]:
plt.figure(figsize=(8,7))
sns.boxplot(x= df["Insulin"], color='red')

In [ ]:
Q1 = df.Insulin.quantile(0.257)
Q3 = df.Insulin.quantile(0.75)
IQR = Q3-Q1
lower = Q1-1.5*IQR
upper = Q3+1.5*IQR
df.loc[df['Insulin']>upper, "Insulin"] = upper

In [ ]:
plt.figure(figsize=(8,7))
sns.boxplot(x= df["Insulin"], color='red')

In [ ]:
# loc
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=10)
lof.fit_predict(df)

In [ ]:
df_scores = lof.negative_outlier_factor_
np.sort(df_scores)[0:20]

In [ ]:
thresold = np.sort(df_scores)[7]

In [ ]:
thresold

In [ ]:
outlier =df_scores>thresold

In [ ]:
df =df[outlier]

In [ ]:
df.head()

Feature Engineering


In [ ]:
NewBMI = pd.Series(["Underweight","Normal","Overweight","Obesity 1","Obesity 2", "Obesity 3",], dtype = "category")

In [ ]:
NewBMI

In [ ]:
df['NewBMI'] = NewBMI
df.loc[df["BMI"]<18.5,"NewBMI"] = NewBMI[0]
df.loc[(df["BMI"]>18.5) & (df["BMI"]<=24.9),"NewBMI"] = NewBMI[1]
df.loc[(df["BMI"]>24.9) & (df["BMI"]<=29.9),"NewBMI"] = NewBMI[2]
df.loc[(df["BMI"]>29.9) & (df["BMI"]<=34.9),"NewBMI"] = NewBMI[3]
df.loc[(df["BMI"]>34.9) & (df["BMI"]<=39.9),"NewBMI"] = NewBMI[4]
df.loc[df["BMI"]>39.5,"NewBMI"] = NewBMI[5]

In [ ]:
df.head()

In [ ]:
# if Insulin>=16 & Insulin<=166--normal
def set_insuline(row):
  if row["Insulin"]>=16 and row["Insulin"]<=166:
    return"Normal"
  else:
    return"Abnormal"

In [ ]:
df = df.assign(NewInsulinscore=df.apply(set_insuline, axis=1))

In [ ]:
df.head()

In [ ]:
NewGlucose = pd.Series(["Low","Normal","Overweight","Secret","High"], dtype = "category")
df["NewGlucose"] = NewGlucose
df.loc[df["Glucose"]<=70,"NewGlucose"] = NewGlucose[0]
df.loc[(df["Glucose"]>70) & (df["Glucose"]<=99),"NewGlucose"] = NewGlucose[1]
df.loc[(df["Glucose"]>99) & (df["Glucose"]<=126),"NewGlucose"] = NewGlucose[2]
df.loc[df["Glucose"]> 126,"NewGlucose"] = NewGlucose[3]

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
categorical_df = df[['NewBMI',
       'NewInsulinscore', 'NewGlucose']]

In [ ]:
categorical_df.head()

In [ ]:
df.columns

In [ ]:
y =df['Outcome']
x=df.drop(['Outcome','NewBMI',
       'NewInsulinscore', 'NewGlucose'], axis=1)

In [ ]:
cols = x.columns
index = x.index

In [ ]:
x.head()

In [ ]:
from sklearn.preprocessing import RobustScaler
transformer = RobustScaler().fit(x)
X=transformer.transform(x)
X=pd.DataFrame(X,columns=cols,index=index)

In [ ]:
X = pd.concat([X,categorical_df],axis=1)

In [ ]:
X.head()

splitting the data

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Convert all string/categorical columns to numeric
x_train = pd.get_dummies(x_train, drop_first=True)
x_test = pd.get_dummies(x_test, drop_first=True)

# Make sure both train and test have same columns
x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

# Now scale
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

Machine learning algorithims

In [ ]:
# Logistic Regression
log_reg = LogisticRegression()
log_reg.fit(x_train,y_train)
y_pred = log_reg.predict(x_test)

In [ ]:
accuracy_score(y_train,log_reg.predict(x_train))

In [ ]:
log_reg_acc = accuracy_score(y_test, log_reg.predict(x_test))

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)  # you can change n_neighbors

knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)
knn_acc = accuracy_score(y_test, knn.predict(x_test))
print(accuracy_score(y_train, knn.predict(x_train)))  # ✅ lowercase 'knn'
print(accuracy_score(y_test, knn.predict(x_test)))
print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test,y_pred))

In [ ]:
# SVM
svc = SVC(probability=True)
parameter = {
    "gamma" :[0.0001,0.001,0.01],
    'C': [0.01,0.05,0.5,0.01,1,10,15,20]
}
grid_search = GridSearchCV(svc,parameter)
grid_search.fit(x_train,y_train)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
svc = SVC(C=10,gamma = 0.01, probability = True)
svc.fit(x_train, y_train)
y_pred = svc.predict(x_test)
svc_acc = accuracy_score(y_test, svc.predict(x_test))
print(accuracy_score(y_train, svc.predict(x_train)))
print(accuracy_score(y_test, svc.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
# Decison tree

In [ ]:
DT = DecisionTreeClassifier()
DT.fit(x_train, y_train)
y_pred = DT.predict(x_test)
DT_acc = accuracy_score(y_test, DT.predict(x_test))
print(accuracy_score(y_train, DT.predict(x_train)))
print(accuracy_score(y_test, DT.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
# hyperparameter tunning on DT
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

DT = DecisionTreeClassifier()

grid_param = {
    'criterion': ['gini', 'entropy'],
    'max_depth': range(5, 8),   # 5, 6, 7
    'splitter': ['best', 'random'],
    'min_samples_leaf': range(1, 6),  # 1 to 5
    'min_samples_split': range(2, 11), # 2 to 10
    'max_features': ['auto', 'sqrt', 'log2']
}

grid_search_dt = GridSearchCV(DT, grid_param, cv=5, n_jobs=-1, verbose=1)
grid_search_dt.fit(x_train, y_train)


In [ ]:
grid_search_dt.best_params_

In [ ]:
grid_search_dt.best_score_

In [ ]:
DT =grid_search_dt.best_estimator_
y_pred = DT.predict(x_test)
DT_acc = accuracy_score(y_test, DT.predict(x_test))
print(accuracy_score(y_train, DT.predict(x_train)))
print(accuracy_score(y_test, DT.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
# Randomforest Classifier
rand_clf = RandomForestClassifier(criterion = 'entropy', max_depth=15, max_features= 0.75,min_samples_leaf= 2, min_samples_split= 3, n_estimators= 130)
rand_clf.fit(x_train, y_train)
y_pred = rand_clf.predict(x_test)

In [ ]:
y_pred = rand_clf.predict(x_test)
rand_clf_acc = accuracy_score(y_test,rand_clf.predict(x_test))
print(accuracy_score(y_train, rand_clf.predict(x_train)))
print(accuracy_score(y_test,rand_clf.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
# Gradienboosting classifier
gbc = GradientBoostingClassifier()
parameters = {
    'loss': ['deviance', 'exponential'],
    'learning_rate': [0.001, 0.1, 1, 10],
    'n_estimators': [100, 150, 180, 200],
}
grid_search_gbc = GridSearchCV(gbc, parameters, cv= 10, n_jobs=-1, verbose=1)
grid_search_gbc.fit(x_train, y_train)

In [ ]:
grid_search_gbc.best_params_

In [ ]:
grid_search_gbc.best_score_

In [ ]:
 gbc = GradientBoostingClassifier(learning_rate=0.1, loss='exponential', n_estimators=150)
gbc.fit(x_train, y_train)

In [ ]:
grid_search_gbc.best_estimator_
y_pred = gbc.predict(x_test)
gbc_acc = accuracy_score(y_test,gbc.predict(x_test))
print(accuracy_score(y_train, gbc.predict(x_train)))
print(accuracy_score(y_test,gbc.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(object = 'binary:logistic', learning_rate = 0.01, max_depth = 10, n_estimators = 180)
xgb.fit(x_train, y_train)


In [ ]:
y_pred = xgb.predict(x_test)
xgb_acc = accuracy_score(y_test,xgb.predict(x_test))
print(accuracy_score(y_train, xgb.predict(x_train)))
print(accuracy_score(y_test,xgb.predict(x_test)))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
# Model Comparison
models = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'SVM', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Score': [100*round(log_reg_acc,4) ,100*round(knn_acc,4), 100*round(svc_acc,4), 100*round(DT_acc, 4), 100*round(rand_clf_acc,4),
              100*round(gbc_acc,4), 100*round(xgb_acc,4)]})
models.sort_values(by = 'Score', ascending = False)


In [ ]:
import pickle

In [ ]:
import pickle

# Put all trained models in a dictionary
saved_models = {
    'Logistic Regression': log_reg,
    'KNN': knn,
    'SVM': svc,
    'Decision Tree': DT,
    'Random Forest': rand_clf,
    'Gradient Boosting': gbc,
    'XGBoost': xgb
}

# Save models
with open('AdvancedDB.pkl', 'wb') as files:
    pickle.dump(saved_models, files)

# Save scaler separately
with open('adb_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ All models saved to AdvancedDB.pkl")
print("✅ Scaler saved to adb_scaler.pkl")


In [ ]:
models = {
   'Logistic Regression': log_reg,
    'KNN': knn,
    'SVM': svc,
    'Decision Tree': DT,
    'Random Forest': rand_clf,
    'Gradient Boosting': gbc,
    'XGBoost': xgb
}



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt

metrics_list = []

for name, model in models.items():

    # Predictions
    y_pred = model.predict(x_test)

    # Try probability else fallback using decision_function
    try:
        y_proba = model.predict_proba(x_test)[:,1]
    except:
        try:
            y_proba = model.decision_function(x_test)
        except:
            y_proba = y_pred  # fallback if nothing available

    metrics_list.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_proba)
    ])

df_metrics = pd.DataFrame(metrics_list, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"
])

df_metrics


In [ ]:
df_metrics.set_index("Model").plot(kind="bar", figsize=(11,6))
plt.xticks(rotation=45)
plt.title("Model Evaluation Comparison")
plt.ylabel("Score")
plt.show()
